<!-- SPDX-License-Identifier: Apache-2.0 -->
# UAIRT with TensorRT on Google Colab

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vnk8071/unified-ai-runtime/blob/main/examples/notebooks/uairt_tensorrt_colab.ipynb)

This notebook runs a model on an NVIDIA GPU through UAIRT's `tensorrt` backend, end to end:

1. install TensorRT and fetch its public headers,
2. build UAIRT and the TensorRT plugin,
3. export ResNet-18 to ONNX and build a TensorRT engine,
4. run the engine through UAIRT's Python binding and **compare the output with PyTorch**,
5. time UAIRT, pinned-memory UAIRT and PyTorch.

**Before you start:** *Runtime → Change runtime type → a GPU* (a free T4 works). Then *Runtime → Run all*.

> **Licences.** Cell 2 installs NVIDIA's TensorRT from its pip packages and cell 3 fetches TensorRT's public headers from
> [github.com/NVIDIA/TensorRT](https://github.com/NVIDIA/TensorRT). Running them means you accept NVIDIA's terms for those downloads;
> UAIRT's repository never contains or redistributes them. Nothing here is committed or uploaded.

> **Versions.** The notebook pins **TensorRT 10.9.0.34 for CUDA 12**, the version UAIRT's TensorRT backend was verified with
> (on a GTX 1650). Other versions may work; newer major versions have not been tried. An engine is tied to the GPU model and the
> TensorRT version it was built with, so build it on the machine that runs it, as this notebook does.

## 1. Check the GPU

In [11]:
import shutil
import subprocess
import sys

if not shutil.which("nvidia-smi"):
    raise SystemExit("No NVIDIA GPU found. In Colab: Runtime -> Change runtime type -> choose a GPU, then run again.")

# The full table: the GPU model, the driver version, and the highest CUDA version that driver supports.
!nvidia-smi

print(subprocess.run(["nvidia-smi", "--query-gpu=name,driver_version,compute_cap,memory.total", "--format=csv"],
                     capture_output=True, text=True).stdout)
print(subprocess.run("nvcc --version | tail -2", shell=True, capture_output=True, text=True).stdout)
print("python", sys.version.split()[0])

Tue Oct 20 09:34:50 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   48C    P0             27W /   70W |     305MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Install TensorRT

`tensorrt-cu12` pulls the libraries and the Python bindings. The pip package has **no C++ headers**, which UAIRT's plugin needs to
compile, so the next cell gets them.

In [12]:
TRT_VERSION = "10.9.0.34"
!pip install -q "tensorrt-cu12=={TRT_VERSION}"

import tensorrt as trt
print("TensorRT", trt.__version__)
assert trt.__version__.startswith("10.9"), "this notebook was written for TensorRT 10.9"

TensorRT 10.9.0.34


## 3. Fetch TensorRT's headers

The headers of the matching release (`v10.9.0`) come from NVIDIA's public repository; only the `include/` directory is checked out.

In [13]:
import os
from pathlib import Path

HEADERS = Path("/content/trt-headers")
if not (HEADERS / "include" / "NvInfer.h").exists():
    subprocess.run(["git", "clone", "-q", "--depth", "1", "--filter=blob:none", "--sparse", "--branch", "v10.9.0",
                    "https://github.com/NVIDIA/TensorRT.git", str(HEADERS)], check=True)
    subprocess.run(["git", "-C", str(HEADERS), "sparse-checkout", "set", "include"], check=True)
TRT_INCLUDE = HEADERS / "include"
print(sorted(p.name for p in TRT_INCLUDE.glob("NvInfer*.h"))[:4], "...")

# The pip package keeps the libraries here; the plugin links libnvinfer.so.10 from this directory.
import tensorrt_libs
TRT_LIBDIR = Path(tensorrt_libs.__file__).parent
TRT_LIBNVINFER = next(TRT_LIBDIR.glob("libnvinfer.so.10*"))
print("libnvinfer:", TRT_LIBNVINFER)

['NvInfer.h', 'NvInferImpl.h', 'NvInferLegacyDims.h', 'NvInferPlugin.h'] ...
libnvinfer: /usr/local/lib/python3.13/dist-packages/tensorrt_libs/libnvinfer.so.10


## 4. Build UAIRT with the TensorRT plugin

This clones UAIRT and builds the shared library, the plugin and `run_model`. It takes a minute or two. The two `UAIRT_TRT_*`
variables tell CMake where the headers and `libnvinfer` are, and the linker flag records the library directory in the plugin.

In [14]:
UAIRT = Path("/content/unified-ai-runtime")
BUILD = Path("/content/uairt-build")
if not UAIRT.exists():
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/vnk8071/unified-ai-runtime.git", str(UAIRT)],
                   check=True)

configure = [
    "cmake", "-S", str(UAIRT), "-B", str(BUILD), "-DCMAKE_BUILD_TYPE=Release", "-DBUILD_SHARED_LIBS=ON",
    "-DUAIRT_BUILD_TESTS=OFF", "-DUAIRT_BUILD_EXAMPLES=OFF", "-DUAIRT_BUILD_TENSORRT=ON",
    f"-DUAIRT_TRT_INCLUDE={TRT_INCLUDE}", f"-DUAIRT_TRT_LIBRARY={TRT_LIBNVINFER}",
    f"-DCMAKE_MODULE_LINKER_FLAGS=-Wl,-rpath,{TRT_LIBDIR}", "-DCUDAToolkit_ROOT=/usr/local/cuda",
]
for step in (configure, ["cmake", "--build", str(BUILD), "-j", str(os.cpu_count() or 2)]):
    done = subprocess.run(step, capture_output=True, text=True)
    print("\n".join(done.stdout.splitlines()[-3:]))
    if done.returncode:
        print(done.stderr[-4000:])
        raise SystemExit(f"{step[0]} failed; read the output above")

plugin = BUILD / "libuairt_backend_tensorrt.so"
assert plugin.exists(), "the plugin was not built"
print("plugin:", plugin)

-- Configuring done (0.1s)
-- Generating done (0.0s)
-- Build files have been written to: /content/uairt-build
[ 60%] Built target uairt
[ 80%] Built target uairt_llm
[100%] Built target uairt_backend_tensorrt
plugin: /content/uairt-build/libuairt_backend_tensorrt.so


## 5. Make a model and a TensorRT engine

ResNet-18 with random weights, so nothing has to be downloaded (swap in `weights="DEFAULT"` for a trained one: the comparison below
works the same). It is exported to ONNX, then TensorRT builds an engine for **this** GPU. UAIRT needs static shapes, so the batch
size is fixed at 1.

In [15]:
# torch.onnx.export needs the onnx package, which Colab does not preinstall.
!pip install -q onnx

import numpy as np
import torch
import torchvision

# Ampere and newer GPUs (A100, L4, ...) compute "FP32" convolutions in TF32 by default, in both PyTorch and TensorRT, which differs from
# true FP32 by about 1e-3. Turn it off in both so the FP32 comparison below is exact on any GPU.
torch.backends.cudnn.allow_tf32 = False
torch.backends.cuda.matmul.allow_tf32 = False

torch.manual_seed(0)
net = torchvision.models.resnet18(weights=None).eval().cuda()
example = torch.zeros(1, 3, 224, 224, device="cuda")

onnx_path = Path("/content/resnet18.onnx")
export_args = dict(input_names=["input"], output_names=["logits"], opset_version=17)
try:
    torch.onnx.export(net, example, str(onnx_path), dynamo=False, **export_args)
except TypeError:                      # older torch has no dynamo argument
    torch.onnx.export(net, example, str(onnx_path), **export_args)
print("ONNX:", onnx_path.stat().st_size // 1024, "KiB")

def build_engine(onnx_file, engine_file, fp16=False):
    logger = trt.Logger(trt.Logger.WARNING)
    builder = trt.Builder(logger)
    network = builder.create_network(0)
    parser = trt.OnnxParser(network, logger)
    if not parser.parse_from_file(str(onnx_file)):
        raise RuntimeError("; ".join(str(parser.get_error(i)) for i in range(parser.num_errors)))
    config = builder.create_builder_config()
    config.set_memory_pool_limit(trt.MemoryPoolType.WORKSPACE, 1 << 30)
    if fp16:
        config.set_flag(trt.BuilderFlag.FP16)
    else:
        config.clear_flag(trt.BuilderFlag.TF32)      # true FP32, to match PyTorch with TF32 off
    blob = builder.build_serialized_network(network, config)
    if blob is None:
        raise RuntimeError("TensorRT could not build the engine")
    Path(engine_file).write_bytes(bytes(blob))

build_engine(onnx_path, "/content/resnet18_fp32.engine")
build_engine(onnx_path, "/content/resnet18_fp16.engine", fp16=True)
print({p.name: p.stat().st_size // 1024 for p in Path("/content").glob("resnet18_*.engine")}, "KiB")

/tmp/ipykernel_1796/744426160.py:20: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(net, example, str(onnx_path), dynamo=False, **export_args)


ONNX: 45638 KiB
{'resnet18_fp16.engine': 23688, 'resnet18_fp32.engine': 56932} KiB


## 6. Run it through UAIRT

`tensorrt` is imported first so `libnvinfer` is already loaded when the plugin needs it. Then `uairt.Engine("tensorrt")` loads the
engine; `model.inputs` and `model.outputs` come from the engine itself, and the input must match them exactly.

In [16]:
os.environ["UAIRT_LIBRARY"] = str(next(BUILD.glob("libuairt.so")))
os.environ["UAIRT_PLUGIN_PATH"] = str(BUILD)
sys.path.insert(0, str(UAIRT / "bindings" / "python"))
import uairt

# available_backends() lists every name the package knows; usable ones are registered or have a plugin file.
usable = sorted(n for n, info in uairt.available_backends().items() if info["registered"] or info["plugin"])
print("UAIRT", uairt.version(), "| usable backends:", usable)
assert "tensorrt" in usable, "the TensorRT plugin was not found; check cell 4"

x = np.random.default_rng(0).random((1, 3, 224, 224), dtype=np.float32)
with torch.no_grad():
    reference = net(torch.from_numpy(x).cuda()).cpu().numpy()

with uairt.Engine("tensorrt") as engine, engine.load_model("/content/resnet18_fp32.engine") as model:
    print("inputs :", [(i.name, i.dtype, i.shape) for i in model.inputs])
    print("outputs:", [(o.name, o.dtype, o.shape) for o in model.outputs])
    (logits,) = model.run(x)

print("max abs difference from PyTorch:", float(np.abs(logits - reference).max()))
np.testing.assert_allclose(logits, reference, rtol=1e-3, atol=1e-3)
assert np.unique(reference).size > 1, "the reference is constant, so the comparison proves nothing"
print("UAIRT + TensorRT (FP32) matches PyTorch; top-1 class", int(logits.argmax()), "=", int(reference.argmax()))

UAIRT 0.2.0 | usable backends: ['onnxruntime', 'reference', 'tensorrt']
inputs : [('input', dtype('float32'), (1, 3, 224, 224))]
outputs: [('logits', dtype('float32'), (1, 1000))]
max abs difference from PyTorch: 1.5497207641601562e-06
UAIRT + TensorRT (FP32) matches PyTorch; top-1 class 238 = 238


## 7. FP16 engine

The FP16 engine keeps float32 inputs and outputs but computes in half precision, so it is faster and a little less exact: the
tolerance is looser.

In [17]:
with uairt.Engine("tensorrt") as engine, engine.load_model("/content/resnet18_fp16.engine") as model:
    (logits16,) = model.run(x)
print("max abs difference from PyTorch (FP16 engine):", float(np.abs(logits16 - reference).max()))
np.testing.assert_allclose(logits16, reference, rtol=5e-2, atol=5e-2)
print("FP16 engine within tolerance; top-1 class", int(logits16.argmax()), "=", int(reference.argmax()))

max abs difference from PyTorch (FP16 engine): 0.002658367156982422
FP16 engine within tolerance; top-1 class 238 = 238


## 8. Timing

Median time per run for UAIRT with ordinary NumPy arrays, UAIRT with **pinned** buffers (page-locked memory, so the copies to and from
the GPU are faster), and PyTorch on the same GPU. The numbers depend on the GPU and on whatever else the Colab VM is doing: treat them
as a rough comparison, not a benchmark.

In [18]:
import time

def median_ms(call, runs=200, warmup=20):
    for _ in range(warmup):
        call()
    samples = []
    for _ in range(runs):
        start = time.perf_counter()
        call()
        samples.append((time.perf_counter() - start) * 1e3)
    return float(np.median(samples))

timings = {}
# The buffers are opened before the models and so closed after them, as the binding requires.
with uairt.Engine("tensorrt") as engine, \
     engine.alloc_buffer(x.nbytes, "pinned") as in_buffer, \
     engine.alloc_buffer(reference.nbytes, "pinned") as out_buffer:
    in_buffer.array(x.dtype, x.shape)[...] = x
    for label, path in (("FP32", "/content/resnet18_fp32.engine"), ("FP16", "/content/resnet18_fp16.engine")):
        with engine.load_model(path) as model:
            timings[f"UAIRT TensorRT {label}"] = median_ms(lambda: model.run(x))
            timings[f"UAIRT TensorRT {label}, pinned buffers"] = median_ms(
                lambda: model.run_buffers([in_buffer], [out_buffer]))

x_gpu = torch.from_numpy(x).cuda()
def torch_run():
    with torch.no_grad():
        net(x_gpu)
    torch.cuda.synchronize()
timings["PyTorch FP32 (eager)"] = median_ms(torch_run)

for label, value in timings.items():
    print(f"{label:<42} {value:8.3f} ms")

UAIRT TensorRT FP32                           1.613 ms
UAIRT TensorRT FP32, pinned buffers           1.455 ms
UAIRT TensorRT FP16                           0.746 ms
UAIRT TensorRT FP16, pinned buffers           0.602 ms
PyTorch FP32 (eager)                          3.114 ms


## 9. The `onnxruntime` backend with its CUDA and TensorRT providers

Sections 1 to 8 used UAIRT's own `tensorrt` backend with an engine file. ONNX Runtime can also run the `.onnx` file on the GPU through its
CUDA and TensorRT **execution providers**, and UAIRT's `onnxruntime` backend selects them with `execution_provider` (see
[docs/backends/onnxruntime.md](https://github.com/vnk8071/unified-ai-runtime/blob/main/docs/backends/onnxruntime.md)).

> **Status.** On a Colab T4 the build and the CUDA provider ran (output below); the TensorRT provider rows have to be run again after a fix. The same options ran
> on a GTX 1650 (CUDA 12.8, cuDNN 9, ONNX Runtime 1.30.0) for the CUDA and TensorRT providers. If a cell fails here, the error text is the
> useful part: please report it.

This cell installs `onnxruntime-gpu`, fetches ONNX Runtime's header (the plugin loads the library at run time and needs only the header to compile), and
rebuilds UAIRT with the `onnxruntime` plugin.

In [19]:
!pip install -q onnxruntime-gpu
import onnxruntime as ort

providers = ort.get_available_providers()
print("onnxruntime", ort.__version__, providers)
assert "CUDAExecutionProvider" in providers, "this onnxruntime-gpu build has no CUDA provider; see Troubleshooting"
ort.preload_dlls()          # loads the CUDA and cuDNN libraries that pip installed, so the plugin's ONNX Runtime finds them too

subprocess.run(["bash", str(UAIRT / "scripts/ci/fetch_onnxruntime_headers.sh")], check=True)
done = subprocess.run(["cmake", "-S", str(UAIRT), "-B", str(BUILD), "-DUAIRT_BUILD_ONNXRUNTIME=ON"], capture_output=True, text=True)
print("\n".join(done.stdout.splitlines()[-2:]) or done.stderr[-800:])
done = subprocess.run(["cmake", "--build", str(BUILD), "-j", str(os.cpu_count() or 2)], capture_output=True, text=True)
print("\n".join(done.stdout.splitlines()[-2:]))
assert done.returncode == 0, done.stderr[-1500:]

ORT_LIBRARY = next((Path(ort.__file__).parent / "capi").glob("libonnxruntime.so.*"))
os.environ["UAIRT_ONNXRUNTIME_LIBRARY"] = str(ORT_LIBRARY)
print("plugin:", next(BUILD.glob("libuairt_backend_onnxruntime.so")), "\nlibrary:", ORT_LIBRARY)

onnxruntime 1.31.0 ['TensorrtExecutionProvider', 'CUDAExecutionProvider', 'CPUExecutionProvider']
-- Generating done (0.0s)
-- Build files have been written to: /content/uairt-build
[100%] Linking C shared module libuairt_backend_onnxruntime.so
[100%] Built target uairt_backend_onnxruntime
plugin: /content/uairt-build/libuairt_backend_onnxruntime.so 
library: /usr/local/lib/python3.13/dist-packages/onnxruntime/capi/libonnxruntime.so.1.31.0


## 10. CPU, CUDA and TensorRT providers on the same `.onnx` file

Each provider runs `resnet18.onnx` from section 5. A line shows how far its output is from PyTorch's and the median time per run.

`cpu_fallback=0` makes the load **fail** if ONNX Runtime would run any node on its CPU provider, so a provider that silently could not start cannot pass
as a GPU result. If a model has a few nodes no provider takes (shape arithmetic is typical), the cell retries without it and says the run is **partly on the CPU**.
The TensorRT provider compiles an engine on first use (a minute or more); the cache keeps later loads short. `trt_fp16_enable` computes in half precision,
so its tolerance is looser, as in section 7.

In [20]:
import time

def median_ms(call, runs=200, warmup=20):
    for _ in range(warmup):
        call()
    samples = []
    for _ in range(runs):
        start = time.perf_counter()
        call()
        samples.append((time.perf_counter() - start) * 1e3)
    return float(np.median(samples))

TRT_CACHE = "/content/trt_cache"
for name in ("fp32", "fp16"):
    os.makedirs(f"{TRT_CACHE}/{name}", exist_ok=True)      # the TensorRT provider does not create its cache directory
variants = [
    ("CPU provider",         {"execution_provider": "cpu"},                                                                    1e-4),
    ("CUDA provider",        {"execution_provider": "cuda", "ep_option.use_tf32": "0"},                                          1e-3),
    ("TensorRT provider",    {"execution_provider": "tensorrt", "ep_option.trt_engine_cache_enable": "1",
                              "ep_option.trt_engine_cache_path": TRT_CACHE + "/fp32"},                                         5e-3),
    ("TensorRT provider, FP16", {"execution_provider": "tensorrt", "ep_option.trt_fp16_enable": "1",
                              "ep_option.trt_engine_cache_enable": "1", "ep_option.trt_engine_cache_path": TRT_CACHE + "/fp16"}, 5e-2),
]

results = {}
for label, options, tolerance in variants:
    placed = "all nodes on the provider"
    strict = options["execution_provider"] != "cpu"
    for attempt in ((True, False) if strict else (False,)):
        opts = dict(options, **({"cpu_fallback": "0"} if attempt else {}))
        try:
            with uairt.Engine("onnxruntime", opts) as engine, engine.load_model(str(onnx_path)) as model:
                (out,) = model.run(x)
                diff = float(np.abs(out - reference).max())
                results[label] = (diff, median_ms(lambda: model.run(x)), placed if (attempt or not strict) else "PARTLY ON THE CPU")
            break
        except Exception as error:
            if attempt and "fallback" in str(error):
                print(f"{label}: strict load refused ({str(error)[:160]}) - retrying without cpu_fallback=0")
                placed = "PARTLY ON THE CPU"
                continue
            results[label] = (None, None, f"failed: {str(error)[:200]}")
    diff, ms, note = results[label]
    if diff is None:
        print(f"{label:<26} {note}")
    else:
        status = "ok" if diff <= tolerance else f"OUTSIDE {tolerance:g}"
        print(f"{label:<26} max diff vs PyTorch {diff:9.2e} ({status})  {ms:8.3f} ms  {note}")

CPU provider               max diff vs PyTorch  2.15e-06 (ok)    34.549 ms  all nodes on the provider
CUDA provider              max diff vs PyTorch  1.67e-06 (ok)     3.187 ms  all nodes on the provider
TensorRT provider          max diff vs PyTorch  1.91e-06 (ok)     1.546 ms  all nodes on the provider
TensorRT provider, FP16    max diff vs PyTorch  2.52e-03 (ok)     0.969 ms  all nodes on the provider


How to read it: a row that says `failed:` with `backend unavailable` means this ONNX Runtime build lacks that provider or its libraries
(for TensorRT, `libnvinfer` of the version the build expects) were not found; it did not run on the CPU instead. A `PARTLY ON THE CPU` row ran, but not
all on the GPU, so its time is not a GPU time. Compare the rows with the `tensorrt` backend's numbers in section 8, which uses a prebuilt engine and pinned buffers.

## Troubleshooting

- **`Module onnx is not installed!` at the export:** cell 5 installs it; if you skipped that line, run `!pip install onnx`.
- **A `libcudart` or CUDA version error when loading or running an engine:** Colab's toolkit can be newer (CUDA 13) than this TensorRT build (CUDA 12). The
  plugin was seen to build in that setup, but running an engine there has not been confirmed. Report the full error so it can be fixed.
- **`nvcc` or `CUDAToolkit` not found at the build step:** use a GPU runtime (a CPU runtime has no CUDA toolkit).
- **`error while loading shared libraries: libnvinfer.so.10`:** import `tensorrt` before `uairt` (cell 6 does), or the plugin was built
  without the `-Wl,-rpath` flag from cell 4.
- **`Unsupported` or `IncompatibleModel` when loading an engine:** the engine was built by a different TensorRT version or for a different
  GPU. Rebuild it in cell 5 on the current runtime; Colab may give you a different GPU each time.
- **Engines with dynamic shapes** are rejected: export with a fixed batch size, as cell 5 does.
- **Another TensorRT version:** change `TRT_VERSION`, the `--branch` tag in cell 3 and the assertion in cell 2 together. Only 10.9 was used
  to verify the backend.

See `docs/backends/tensorrt.md` for the backend's options and limits, and `examples/python/detect_tensorrt.py` for a YOLOv8
engine with pinned buffers.

- **`CUDAExecutionProvider` missing or `libcudnn` / `libcublas` not found in section 9:** `onnxruntime-gpu` targets one CUDA major version; if Colab's
  differs, install the build for it (see ONNX Runtime's CUDA requirements) and restart the runtime. `ort.preload_dlls()` only loads libraries that pip installed.
- **`TensorRT provider` fails with `backend unavailable`:** the ONNX Runtime build expects a `libnvinfer` major version that is not on the library path; the pip `tensorrt-cu12` libraries from section 2 are in `TRT_LIBDIR`.
